In [1]:
%%capture
pip install transformer_lens transformers jaxtyping

In [2]:
import torch
import functools
#import einops
import numpy as np
#import pandas as pd  

#from sklearn.model_selection import train_test_split
from tqdm import tqdm
from torch import Tensor
from typing import List, Callable
from transformer_lens import HookedTransformer, utils
from transformer_lens.hook_points import HookPoint
from transformers import AutoModelForCausalLM, AutoTokenizer
from jaxtyping import Float, Int

/opt/homebrew/Caskroom/miniconda/base/envs/algo-neutrality/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
def getDevice():
    if torch.cuda.is_available(): #nvidia/runpod
        return torch.device("cuda")
    elif torch.backends.mps.is_available():
        return torch.device("mps") #apple silicon
    else:
        return torch.device("cpu") #not recommended
    
DEVICE = getDevice()
DEVICE

device(type='mps')

In [7]:
model_name = "Qwen/Qwen1.5-1.8B-Chat"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto")

Some parameters are on the meta device because they were offloaded to the disk.


In [8]:
prompt = "Answer the following question in French: Who was the first president of USA?"

messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": prompt}
]

chat_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

inputs = tokenizer(chat_prompt, return_tensors="pt", padding=True)
inputs = {k: v.to(model.device) for k, v in inputs.items()}


# Generate
output_ids = model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.7,
    do_sample=True,
    top_p=0.9
)

response = tokenizer.decode(output_ids[0], skip_special_tokens=True)
print(response)

KeyboardInterrupt: 